In [3]:
import pandas as pd

df = pd.read_csv("../data/city_day.csv")
df.head()


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


In [4]:
import pandas as pd
pd.options.display.max_columns = 200

df = pd.read_csv("../data/city_day.csv")
print("Shape:", df.shape)
display(df.head(5))


Shape: (29531, 16)


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


In [5]:
# normalize columns
df.columns = [c.strip() for c in df.columns]
col_map = {
    "City": "city",
    "Date": "datetime",
    "PM2.5": "pm25",
    "PM10": "pm10",
    "NO": "no",
    "NO2": "no2",
    "NOx": "nox",
    "NH3": "nh3",
    "CO": "co",
    "SO2": "so2",
    "O3": "o3",
    "AQI": "aqi",
    "AQI_Bucket": "aqi_bucket",
    "Benzene": "benzene",
    "Toluene": "toluene",
    "Xylene": "xylene"
}
# apply mapping where keys exist
df = df.rename(columns={k:v for k,v in col_map.items() if k in df.columns})
print("New columns:", df.columns.tolist())


New columns: ['city', 'datetime', 'pm25', 'pm10', 'no', 'no2', 'nox', 'nh3', 'co', 'so2', 'o3', 'benzene', 'toluene', 'xylene', 'aqi', 'aqi_bucket']


In [6]:
# parse datetime
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
print("Null datetimes:", df['datetime'].isna().sum())

# drop rows with bad datetime (if any)
df = df.dropna(subset=['datetime']).reset_index(drop=True)

# lower-case city names and strip spaces
df['city'] = df['city'].astype(str).str.strip()

# convert pollutant columns to numeric (coerce errors -> NaN)
pollutant_cols = ['pm25','pm10','no','no2','nox','nh3','co','so2','o3','benzene','toluene','xylene','aqi']
for c in pollutant_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# quick type check
display(df.dtypes)


Null datetimes: 0


city                  object
datetime      datetime64[ns]
pm25                 float64
pm10                 float64
no                   float64
no2                  float64
nox                  float64
nh3                  float64
co                   float64
so2                  float64
o3                   float64
benzene              float64
toluene              float64
xylene               float64
aqi                  float64
aqi_bucket            object
dtype: object

In [7]:
# missing summary
miss = df.isna().sum().sort_values(ascending=False)
display(miss[miss>0])

# rows per city (helps check coverage)
rows_city = df.groupby('city')['datetime'].count().sort_values(ascending=False)
display(rows_city.head(20))


xylene        18109
pm10          11140
nh3           10328
toluene        8041
benzene        5623
aqi            4681
aqi_bucket     4681
pm25           4598
nox            4185
o3             4022
so2            3854
no2            3585
no             3582
co             2059
dtype: int64

city
Ahmedabad             2009
Bengaluru             2009
Chennai               2009
Mumbai                2009
Lucknow               2009
Delhi                 2009
Hyderabad             2006
Patna                 1858
Gurugram              1679
Visakhapatnam         1462
Amritsar              1221
Jorapokhar            1169
Jaipur                1114
Thiruvananthapuram    1112
Amaravati              951
Brajrajnagar           938
Talcher                925
Kolkata                814
Guwahati               502
Coimbatore             386
Name: datetime, dtype: int64

In [8]:
# ensure sorted
df = df.sort_values(['city','datetime']).reset_index(drop=True)

# forward/backward fill small gaps per city (limit 3 days)
df = df.groupby('city').apply(lambda g: g.ffill(limit=3).bfill(limit=3)).reset_index(drop=True)

# show remaining missing after short fills
miss2 = df.isna().sum().sort_values(ascending=False)
display(miss2[miss2>0])


C:\Users\shree\AppData\Local\Temp\ipykernel_19552\439550587.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('city').apply(lambda g: g.ffill(limit=3).bfill(limit=3)).reset_index(drop=True)


xylene        17544
pm10          10368
nh3            9574
toluene        7278
benzene        4730
pm25           3869
nox            3619
aqi            3486
aqi_bucket     3486
o3             2989
so2            2879
no2            2739
no             2734
co             1238
dtype: int64

In [9]:
# choose source for labels
if 'aqi' in df.columns and df['aqi'].notna().sum() > 0:
    print("Using 'aqi' to create risk labels.")
    source_col = 'aqi'
elif 'pm25' in df.columns and df['pm25'].notna().sum() > 0:
    print("No AQI found — using 'pm25' as proxy for risk labels.")
    df['aqi'] = df['pm25']  # create proxy
    source_col = 'aqi'
else:
    raise ValueError("Neither 'aqi' nor 'pm25' contain usable values to create risk labels.")

# define bins (you can change thresholds to CPCB later)
bins = [-1, 100, 200, 1e9]
labels = ['Low','Moderate','High']
df['risk'] = pd.cut(df[source_col], bins=bins, labels=labels)

# show distribution
display(df['risk'].value_counts(dropna=False))


Using 'aqi' to create risk labels.


risk
Low         10052
Moderate     9235
High         6758
NaN          3486
Name: count, dtype: int64

In [10]:
keep = ['datetime','city','aqi','pm25','pm10','no','no2','nox','nh3','co','so2','o3','benzene','toluene','xylene','risk']
keep = [c for c in keep if c in df.columns]
df_model = df[keep].copy()

out_path = "../data/city_day_cleaned.csv"
df_model.to_csv(out_path, index=False)
print("Saved cleaned data to:", out_path)
print("Final shape:", df_model.shape)
display(df_model.head(5))


Saved cleaned data to: ../data/city_day_cleaned.csv
Final shape: (29531, 16)


,datetime,city,aqi,pm25,pm10,no,no2,nox,nh3,co,so2,o3,benzene,toluene,xylene,risk
0,2015-01-01,Ahmedabad,NaN,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN
1,2015-01-02,Ahmedabad,NaN,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN
2,2015-01-03,Ahmedabad,NaN,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN
3,2015-01-04,Ahmedabad,NaN,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN
4,2015-01-05,Ahmedabad,NaN,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN
